# Fetching data at warp speed

In [1]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["AUD/USD", "NZD/USD", "EUR/NOK", "EUR/SEK", "GBP/USD", "EUR/USD"]
MONTHS = [
    "202310", "202311", "202312"
]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

Mounted at /content/drive
🚀 Starting Parallel Download of 18 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2023-11-01T02:31:25.708000
INFO:DUKASCRIPT:current timestamp :2023-10-02T04:42:55.629000
INFO:DUKASCRIPT:current timestamp :2023-10-02T03:56:16.943000
INFO:DUKASCRIPT:current timestamp :2023-12-01T05:15:02.665000
INFO:DUKASCRIPT:current timestamp :2023-12-01T11:48:36.652000
INFO:DUKASCRIPT:current timestamp :2023-10-02T07:04:53.378000
INFO:DUKASCRIPT:current timestamp :2023-11-01T05:50:05.756000
INFO:DUKASCRIPT:current timestamp :2023-10-02T08:07:15.024000
INFO:DUKASCRIPT:current timestamp :2023-12-01T16:25:38.754000
INFO:DUKASCRIPT:current timestamp :2023-10-02T09:00:43.753000
INFO:DUKASCRIPT:current timestamp :2023-11-01T11:24:43.083000
INFO:DUKASCRIPT:current timestamp :2023-10-02T12:17:26.781000
INFO:DUKASCRIPT:current timestamp :2023-12-03T23:32:21.443000
INFO:DUKASCRIPT:current timestamp :2023-10-02T10:51:36.055000
INFO:DUKASCRIPT:current timestamp :2023-11-01T15:24:31.548000
INFO:DUKASCRIPT:current timestamp :2023-10-02T13:15:16.564000
INFO:DUK

⏭️ Skipped (Already Exists): AUD/USD 202310
⏭️ Skipped (Already Exists): AUD/USD 202311
⏭️ Skipped (Already Exists): AUD/USD 202312
⏭️ Skipped (Already Exists): NZD/USD 202310
⏭️ Skipped (Already Exists): NZD/USD 202311
⏭️ Skipped (Already Exists): NZD/USD 202312
✅ Success: EUR/NOK 202310 -> eurnok_dukascopy_bid_202310.parquet (3,767,997 rows) & eurnok_dukascopy_ask_202310.parquet (3,767,997 rows)
✅ Success: EUR/NOK 202311 -> eurnok_dukascopy_bid_202311.parquet (2,803,471 rows) & eurnok_dukascopy_ask_202311.parquet (2,803,471 rows)
✅ Success: EUR/NOK 202312 -> eurnok_dukascopy_bid_202312.parquet (2,581,805 rows) & eurnok_dukascopy_ask_202312.parquet (2,581,805 rows)
✅ Success: EUR/SEK 202310 -> eursek_dukascopy_bid_202310.parquet (3,659,601 rows) & eursek_dukascopy_ask_202310.parquet (3,659,601 rows)
✅ Success: EUR/SEK 202311 -> eursek_dukascopy_bid_202311.parquet (2,734,030 rows) & eursek_dukascopy_ask_202311.parquet (2,734,030 rows)
✅ Success: EUR/SEK 202312 -> eursek_dukascopy_bid_2